## 47_import_tables  (NEW workspace) — Tables/Schema Migration step 2 (stream T)

Lifts the OLD schema + table names into the NEW catalog — the step a naive
migration forgets. Schema and table names are preserved verbatim; the catalog
is the ONLY new component:
  old: schema.table                 e.g. practice.practice_customers
  new: TARGET_CATALOG.schema.table  e.g. practice_ml_dev.practice.practice_customers
  1. Unpack tables_export.zip from the staging volume
  2. CREATE SCHEMA IF NOT EXISTS <TARGET_CATALOG>.<OLD_SCHEMA>  <- the OLD
     schema, created inside the new catalog
  3. CREATE OR REPLACE TABLE ... DEEP CLONE delta.`<snapshot>`  <- OLD names
     (CTAS fallback if a runtime rejects cloning from a path)
  4. Row-count parity vs the export inventory + old->new mapping + UUIDs
PREREQ: 45_setup_new_ws run; tables_export.zip uploaded into the volume.
TARGET_CATALOG: auto (Option A) | practice_ml_dev | practice_ml_stg | practice_ml_prd


In [ ]:

%run ./lib/config_lib


In [ ]:

dbutils.widgets.text("TARGET_CATALOG", "practice_ml_dev", "Target catalog — created + named by 45 (practice_ml_dev / _stg / _prd); 'auto' = detect")
dbutils.widgets.text("OLD_SCHEMA", "practice", "OLD schema name (kept identical after migration)")
dbutils.widgets.text("CATALOG", "practice_ml", "Landing-zone catalog (zip location)")
dbutils.widgets.text("SCHEMA", "models", "Landing-zone schema (zip location)")
dbutils.widgets.text("VOLUME", "staging_artifacts", "Landing-zone volume (zip location)")
dbutils.widgets.text("TABLES_ZIP", "tables_export.zip", "Uploaded zip file name")


In [ ]:

import json, os, shutil
import pandas as pd

TARGET_CATALOG = dbutils.widgets.get("TARGET_CATALOG").strip()
OLD_SCHEMA = dbutils.widgets.get("OLD_SCHEMA").strip()
ZONE_CATALOG = dbutils.widgets.get("CATALOG").strip()
ZONE_SCHEMA = dbutils.widgets.get("SCHEMA").strip()
ZONE_VOLUME = dbutils.widgets.get("VOLUME").strip()
ZIP_NAME = dbutils.widgets.get("TABLES_ZIP").strip()

VOL_ROOT = f"/Volumes/{ZONE_CATALOG}/{ZONE_SCHEMA}/{ZONE_VOLUME}"
ZIP_PATH = f"{VOL_ROOT}/{ZIP_NAME}"
UNPACK_DIR = f"{VOL_ROOT}/tables_export"
assert os.path.exists(ZIP_PATH), (
    f"{ZIP_PATH} not found. Upload {ZIP_NAME} into the volume via Catalog Explorer "
    f"(see runbook Phase 2)."
)

shutil.rmtree(UNPACK_DIR, ignore_errors=True)
shutil.unpack_archive(ZIP_PATH, UNPACK_DIR)

with open(f"{UNPACK_DIR}/tables_list.json") as f:
    inventory = json.load(f)
old_schema = inventory.get("old_schema") or OLD_SCHEMA
tables = inventory.get("tables", [])
assert tables, "Inventory is empty — re-run 35_export_tables in the OLD workspace."
print(f"Inventory: {len(tables)} table(s) exported from OLD schema '{old_schema}'")


In [ ]:

# Build the CFG from the INVENTORY's old schema (the widget is only a fallback).
CFG = setup_config(
    config_path="",
    run_date="all",
    catalog=TARGET_CATALOG,
    schema=old_schema,
)
print(f"Target catalog : {CFG.catalog if CFG.catalog else '(auto/classic -> 2-part names)'}")
print(f"OLD schema     : {CFG.schema}  (unchanged — only the catalog is new)")

# Step 1 — THE step a naive migration forgets: create the OLD schema name
# inside the NEW catalog (idempotent; 45 may have created it already).
if CFG.catalog is None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CFG.schema}")
    schema_loc = CFG.schema
else:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CFG.catalog}.{CFG.schema}")
    schema_loc = f"{CFG.catalog}.{CFG.schema}"
print(f"OLD schema created in the NEW workspace: {schema_loc}")


In [ ]:

# Step 2 — DEEP CLONE each table with its OLD name (CTAS fallback per table).
results = []
for entry in tables:
    t = str(entry["table"])
    expected = int(entry["row_count"])
    src = f"{UNPACK_DIR}/data/{t}"
    tgt = CFG.table(t)
    method = "DEEP CLONE"
    try:
        spark.sql(f"CREATE OR REPLACE TABLE {tgt} DEEP CLONE delta.`{src}`")
    except Exception as e:
        print(f"DEEP CLONE failed for {t} ({str(e)[:100]}...) — CTAS fallback")
        spark.read.format("delta").load(src).write.format("delta").mode("overwrite").saveAsTable(tgt)
        method = "CTAS fallback"
    n = spark.read.table(tgt).count()
    ok = "PASS" if n == expected else "FAIL"
    results.append({"old": f"{old_schema}.{t}", "new": tgt,
                    "rows": n, "expected": expected, "method": method, "status": ok})
    print(f"{ok}  {old_schema}.{t}  ->  {tgt}  ({n}/{expected} rows, {method})")

display(pd.DataFrame(results))
assert all(r["status"] == "PASS" for r in results), "Row-count parity failed — check the output above"


In [ ]:

# Step 3 — old -> new mapping + the §6.2 UUID gotcha (every DEEP CLONE assigns a
# NEW table UUID; streaming checkpoints seeded against the old one must be reseeded).
print("=== OLD -> NEW MAPPING (schema + table names UNCHANGED; only the catalog is new) ===")
for r in results:
    print(f"  {r['old']:<42} ->  {r['new']}")
print()
print("=== NEW TABLE UUIDs (guideline §6.2 streaming-checkpoint gotcha) ===")
for r in results:
    detail = spark.sql(f"DESCRIBE DETAIL {r['new']}").collect()[0]
    print(f"  {r['new']}  uuid={detail['id']}")

print("\nT-STREAM DONE — OLD schema + tables lifted into the new catalog.")
print("Update MIGRATION_CHECKLIST.md rows T1-T4. Then run 50_register_models / 70_import_jobs (or the bundle).")
